Задолбался я искать где посмотреть аниме, искать где есть нужная озвучка и т.п.

Вот проект помогающий с этой проблемой

# Anime Watcher

Для начала надо определить стандартные объекты.

Воспользуемся модулями `dataclass` из библиотеки `pydantic` для составления объекта, модулем `Enum` из библиотеки `enum` для составления категорий и модулем `Path` из библиотеки `pathlib` для загруженного медиаконтента.

## Основные объекты

Основной объект релиза `Anime` должен содержать атрибуты:
- `type`, означающий тип релиза.
- `title`, содержащий локализованный заголовок релиза.
- `description`, содержащий локализованное описание релиза.
- `preview`, содержащий превью релиза.
- `trailer`, содержащий ссылку на трейлер релиза.
- `ratings`, содержащий рейтинг релиза в доступных источниках.
- `episodeCount`, означающий количество эпизодов в релизе.
- `episodes`, содержащий список эпизодов
- `genres`, содержащий жанры аниме
- `tags`, содержащий теги
- `status`, означающий статус аниме - вышло оно, онгоинг или только планируется.
- `favorite`, означающий отмечено ли аниме как избранное

Объект эпизода `AnimeEpisode`:
- `title`, содержащий локализованный заголовок эпизода.
- `description`, содержащий локализованное описание эпизода.
- `langs`, список содержащий доступные языки
- `voices`, список содержащий доступные озвучки/субтитры.

In [13]:
from typing import Union
from enum import Enum, auto
from datetime import datetime
from pathlib import Path
from pydantic import Field
from pydantic.dataclasses import dataclass
from pydantic.networks import HttpUrl


class Language(Enum):
    """ Доступные языки """
    RUSSIAN: int = auto()
    ENGLISH: int = auto()
    JAPANESE: int = auto()


class AnimeType(Enum):
    """ Тип аниме-релиза """
    FILM: int = auto()  # Полнометражный аниме фильм
    SHORT: int = auto()  # Короткометражный аниме фильм
    SERIAL: int = auto()  # Аниме-сериал


class LocaleType(Enum):
    """ Источник локализации """
    IMPORTED: int = auto()  # Импортировано из сторонних хранилищ
    TRANSLATED: int = auto()  # Переведено автоматически


class Status(Enum):
    """ Статус выхода релиза """
    PREMIERE: int = auto()
    ONGOING: int = auto()
    RELEASED: int = auto()


@dataclass
class Localized:
    """ Локализированный тектовый контент """
    lang: Language
    content: str
    type: LocaleType


class PlayerType(Enum):
    """ Тип видеоплеера """
    INTERNAL: int = auto()
    EXTERNAL: int = auto()


@dataclass
class Rating:
    """ Рейтинг релиза в конкретном источнике """
    source: str
    value: float


@dataclass
class Player:
    """ Плеер для аниме эпизода """
    type: PlayerType
    resolution: str
    src: Union[HttpUrl,Path]
    

@dataclass
class Genre:
    """ Жанры """
    id: int
    text: str


@dataclass
class Tag:
    """ Тэги """
    id: int
    text: str


@dataclass
class AnimeVoice:
    """ Доступная озвучка для эпизода """
    title: str
    lang: Language
    players: list[Player] = Field(default_factory=list)


@dataclass
class AnimeEpisode:
    """ Эпизод (отдельная серия/сам фильм) содержащий классы для воспроизведения. """
    title: Localized
    description: Localized
    langs: list[Language] = Field(default_factory=list)
    voices: list[AnimeVoice] = Field(default_factory=list)


@dataclass
class Anime:
    """ Класс, определяющий объект аниме фильма, сериала и т.п. """
    type: AnimeType
    title: Localized
    description: Localized
    preview: Path
    episodeCount: int
    status: Status
    startdate: datetime
    enddate: datetime
    favorite: bool = False
    tags: list[Tag] = Field(default_factory=list)
    genres: list[Genre] = Field(default_factory=list)
    episodes: list[AnimeEpisode] = Field(default_factory=list)
    ratings: list[Rating] = Field(default_factory=list)
    trailer: Union[HttpUrl,Path,None] = None

## Объекты для сторонних сервисов

Также необходимо описать объекты, необходимые для стандартизации общения со сторонними сервисами.

Необходимы следующие объекты:
- `TitleEntry`, заголовки аниме и их локализации
- `AnimeEntry`, объект релиза, готовый к обработке

In [12]:
@dataclass
class TitleEntry:
    animeID: int
    langs: list[Language] = Field(default_factory=list)


@dataclass
class AnimeEntry(Anime):
    preview: HttpUrl
    titles: list[Localized] = Field(default_factory=list)

Необходимо определить требования к API клиенту.

In [ ]:
from typing import Protocol
import requests

class AniClient(Protocol):
    @property
    def baseUrl(self) -> HttpUrl:
        ...
    def browse(self) -> Generator[list[AnimeEntry], None, None]:
        ...
    def getRelease(self, id: str) -> Generator[AnimeEntry, None, None]:
        ...

class Anilibria(AniClient):
    class Anitypes(Enum):
        FILM = 0
        TV = auto()
        ONA = auto()
        OVA = auto()
        SPECIAL = auto()
    @property
    def baseUrl(self):
        return "https://api.anilibria.tv/v3"
    def browse(self):
        for anitype in Anitypes:
            url = f"{self.baseUrl}/title/search"
            params = {
                "items_per_page": 100,
                "type": anitype.value
            }
            response = requests.get(url, params=params)
            data = response.json()
            